In [1]:
!pwd

/Users/rarko/Git/Projects/DQT/etp-dqt


# `ds-monitoring`

[DS Reporting Confluence](https://arrivelogistics.atlassian.net/wiki/spaces/DS/pages/2437414936/ETP+Monitoring+Dashboards)

In [2]:
from loguru import logger
from pathlib import Path

import numpy as np
import polars as pl
import pandas as pd
import altair as alt


from arriveds.snowflake import query_sf

from dqt import is_fresh, get_dt_local, stringify_dates, resolve_data_dir
from dqt.score.constants import COST_COL, TIME_COL, ID_COL, DATE_COL, QUANTILES
from dqt.score.metrics import pinball_usd, mae_usd, mape_pct

DATA_DIR = resolve_data_dir()

/Users/rarko/Git/Projects/DQT/etp-dqt/.venv/lib/python3.12/site-packages/snowflake/connector/vendored/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.7.0) or chardet (7.6.0)/charset_normalizer (3.5.1) doesn't match a supported version!
  warnings.warn(


In [3]:
## Local .parquet data lake settings
COMPRESSION = "zstd"

PATH_CACHE = Path("data/etp")
PATH_CACHE.mkdir(exist_ok=True, parents=True)

PATH_DS_QUERIES = Path("SQL") / "ds-monitoring-queries"
[f.name for f in sorted(PATH_DS_QUERIES.glob("*.sql"))]

['dqt_alt_percentiles.sql',
 'dqt_vs_non_adjusted.sql',
 'etp_dashboard_v2.sql',
 'etp_monitoring_monthly_report.sql']

In [4]:
DASHBOARD_DF_MAP = {
    "DashboardV2": "etp_dashboard_v2.sql",
    "DQTvsNonAdjusted": "dqt_vs_non_adjusted.sql",
    "MonthlyReport": "etp_monitoring_monthly_report.sql",
    "DQTAltPercentiles": "dqt_alt_percentiles.sql",
}

In [6]:
name_query = 'etp_monitoring_monthly_report.sql'
name_query = 'etp_dashboard_v2.sql'
name_query = 'dqt_vs_non_adjusted.sql'
# name_query = 'dqt_alt_percentiles.sql'

path_query = PATH_DS_QUERIES / Path(name_query)
sql_query = path_query.read_text()
path_cache = PATH_CACHE / name_query.replace(".sql", ".parquet")

title = next(iter({k:v for k,v in DASHBOARD_DF_MAP.items() if v == name_query}.keys()))

if is_fresh(path_cache):
    df = pl.read_parquet(path_cache)
    df_schema = df.schema
    logger.success(f"Read {df.height:,d} rows and {len(df_schema):,d} columns from {path_cache}")
else:
    # logger.debug(sql_query)
    logger.info(f"reading {path_query.name} from Snowflake...")
    df = query_sf(sql_query)
    df.write_parquet(path_cache, compression=COMPRESSION)
    logger.info(f"Saved {df.height:,d} rows to {path_cache}")

2026-08-26 13:49:04.522 | SUCCESS  | __main__:<module>:15 - Read 2,980,936 rows and 122 columns from data/etp/dqt_vs_non_adjusted.parquet


In [7]:
from dqt.model.global_dqt import GlobalDQT


def plot_dqt_daily_dial(df: pl.DataFrame):
    base = alt.Chart(df).mark_line().encode(
        x="date:T",
        y="alt_50:Q",
    )
    hline = alt.Chart(pd.DataFrame({"y": [0.50]})).mark_rule(
        color="red",
        strokeDash=[5, 5],
    ).encode(y="y:Q")
    fig = (base + hline).properties(title="DQT Daily Global Dial").configure_title(
        fontSize=20, font="Arial", anchor="start"
    )
    fig.show()


GDQT = GlobalDQT()
daily_global_dial = GDQT.history()  # cached under DQT_DATA_DIR; alias: read_latest_daily_offsets()
today = GDQT.latest()
print(f"latest dial {today.valid_date}: alt_50={today.alt_50:.4f}")
plot_dqt_daily_dial(daily_global_dial)
GDQT.diagnostics(daily_global_dial).select(
    "valid_date", "alt_50", "alt50_off_pp", "alt50_dod_pp"
).tail(10)


2026-08-26 13:49:09.315 | INFO     | dqt.model.global_dqt:_load_history:923 - fetching dial history via dqt_alt_percentiles.sql
2026-08-26 13:49:10.865 | INFO     | arriveds.snowflake.utils:query_sf:422 - dqt_alt_percentiles.sql: 938 rows x 24 cols in 0.90s (query_id=01c6a749-0b15-3e4b-0000-94e586de942e)


latest dial 2026-08-26: alt_50=0.4370


alt.LayerChart(...)

valid_date,alt_50,alt50_off_pp,alt50_dod_pp
date,f64,f64,f64
2026-08-17,0.421517,-7.84825,0.143861
2026-08-18,0.42221,-7.778971,0.069279
2026-08-19,0.423217,-7.678282,0.100689
2026-08-20,0.424499,-7.550138,0.128144
2026-08-21,0.426531,-7.346859,0.20328
2026-08-22,0.428545,-7.145535,0.201324
2026-08-23,0.430603,-6.939733,0.205802
2026-08-24,0.432306,-6.769405,0.170328
2026-08-25,0.43441,-6.55898,0.210425


In [8]:
# if "snowflakeupdatedon" in df.columns:
#     df = df.with_columns(pl.col("snowflakeupdatedon").cast(pl.Date).alias("date"))
# df

In [9]:
if title in ["MonthlyReport", "DashboardV2"]:
    df = df.with_columns(pl.col("booked_on_cst").cast(pl.Date).alias(DATE_COL))
elif title == "DQTvsNonAdjusted":
    df = df.with_columns(pl.col("event_timestamp_utc").cast(pl.Date).alias(DATE_COL))

daily_cnt = df.group_by(DATE_COL).agg(pl.col(ID_COL).n_unique().alias("shipments")).sort(DATE_COL)
fig = daily_cnt.plot.line(x="date", y="shipments")
fig.show()

alt.Chart(...)

In [10]:
# df.select("is_bad_bounce").mean().collect()

In [11]:
# ix_bad_bounce = df.filter(pl.col("is_bad_bounce") > 0)
# df.filter(ix_bad_bounce)

In [12]:
df.group_by("is_bad_bounce").agg(
    pl.col("loadnumber").n_unique().alias("shipments"),
    pl.col("totalcosts").mean().alias("avg_cost"),
    pl.col("totalcosts_less_accessorials").mean().alias("avg_cost_net"),
    # pl.col("loadmiles").mean()
)

is_bad_bounce,shipments,avg_cost,avg_cost_net
i8,u32,f64,f64
0,2780145,1485.347173,1458.081981
1,200791,1525.954242,1498.104376


In [13]:
etp_vs_dqt = df.clone()
cols = etp_vs_dqt.columns

original_qcols = [c for c in cols if c.startswith("orig_")]
adjusted_qcols = [c for c in cols if c.startswith("alt_")]
final_qcols = [c for c in cols if c.startswith("final_")]

In [14]:
_quantile = .50
ix_quantile = [i for i, q in enumerate(QUANTILES) if q == _quantile * 100][0]

COST_COL = "totalcosts_less_accessorials"
MIN_COST = 200

df_cost_null = df.filter(pl.col(COST_COL).is_null())
df_cost_too_low = df.filter(pl.col(COST_COL) <= MIN_COST)
df_missing_q = df.filter(pl.any_horizontal(pl.col(original_qcols).is_null()))

rm_ids = pl.concat([df_cost_too_low.select(ID_COL), df_missing_q.select(ID_COL), df_cost_null.select(ID_COL)])
logger.info(f"Removing {rm_ids.height:,d} rows with cost <= {MIN_COST} or missing quantile predictions or cost data")
df = df.join(rm_ids, on=ID_COL, how="anti")

2026-08-26 13:49:12.720 | INFO     | __main__:<module>:12 - Removing 191,618 rows with cost <= 200 or missing quantile predictions or cost data


In [15]:
# features.filter(pl.col("load_type").is_in(("DRY", "REEFER")))

# from dqt.panel import EQUIPMENT_MAP

# EQUIP_TYPES = list(EQUIPMENT_MAP.keys())
# EQUIP_TYPES

In [16]:
# y = df[COST_COL].to_pandas().values.astype(np.float64)
# # y = df[COST_COL].to_numpy().ravel()
# # ts = df[TIME_COL]
# # ds = df[DATE_COL]

# Q_orig = df[original_qcols]
# Q_alt = df[adjusted_qcols]
# Q_final = df[final_qcols]

# X_o = Q_orig.to_numpy()
# X_a = Q_alt.to_numpy()
# X_f = Q_final.to_numpy()
# # X_o

# y_q = X_o[:, ix_quantile]
# y_q_adj = X_a[:, ix_quantile]
# y_q_final = X_f[:, ix_quantile]
# pb_q_orig = pinball_usd(y_q, y, level=.5)
# pb_q_adj = pinball_usd(y_q_adj, y, level=.5)
# pb_q_final = pinball_usd(y_q_final, y, level=.5)

# logger.info(f"Pinball loss (orig): {pb_q_orig:.2f}")
# logger.info(f"Pinball loss (adj): {pb_q_adj:.2f}")
# logger.info(f"Pinball loss (final): {pb_q_final:.2f}")

In [17]:
### DQT Analysis Shipments
dff = pl.read_parquet(DATA_DIR / "features.parquet")
cols = df.columns

is_cols = [c for c in cols if c.startswith("is_")]
df_bb = df.filter(pl.col("is_bad_bounce") == 1)
df = df.filter(pl.col("is_etp_eligible") == 1, pl.col("is_bad_bounce") == 0)

In [ ]:
path_query = Path("SQL/etp-slider/etp-slider-shift.sql")
df_slides = query_sf(path_query)


In [ ]:
df_slides
